In [1]:
import os
os.add_dll_directory('C:\\Program Files\\IBM\\SQLLIB\\BIN')
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from queries import get_socioeconomico
import ibm_db
import pandas as pd
pd.set_option('display.max_columns', None)

#### Historico

In [3]:
df_datos_hist = pd.read_csv("data/desercion/datos_hist.csv")

In [111]:
df_datos_hist.head()

,ANIO,TERMINO,IDPERIODO,COD_ESTUDIANTE,COD_MATERIA_ACAD,PROMEDIO,VEZ_TOMADA,ESTADO_MAT_TOMADA,ANIO_INGRESO,TERMINO_INGRESO
0,2017,1S,564,201614807,ACUG1003,6.00,1,AP,2016,2S
1,2017,1S,564,201413602,ACUG1001,5.74,1,AN,2014,2S
2,2017,1S,564,201248595,ACUG1001,6.01,1,AP,2012,2S
3,2017,1S,564,201314211,ACUG1001,6.03,1,AP,2013,2S
4,2017,1S,564,201408619,ACUG1001,6.06,1,AP,2014,1S


In [120]:
df_datos_hist["ANIO_INGRESO"].value_counts(sort=True)

ANIO_INGRESO
2019    73109
2015    71141
2016    63972
2017    62611
2018    58528
2014    55315
2020    37854
2013    26093
2012    16227
2011    11188
2010     4282
2008     4233
2007     3432
2009     2755
2006     2433
2005     1501
2004     1488
2003     1345
2002     1246
2001     1064
1999      920
1998      530
2000      296
1997      229
1996      150
1992       69
1989       31
1995       20
1988        1
Name: count, dtype: int64

In [122]:
df_datos_hist[df_datos_hist["ANIO_INGRESO"] == 1988]

,ANIO,TERMINO,IDPERIODO,COD_ESTUDIANTE,COD_MATERIA_ACAD,PROMEDIO,VEZ_TOMADA,ESTADO_MAT_TOMADA,ANIO_INGRESO,TERMINO_INGRESO
2826,2020,1S,574,198802423,ACUG1058,9.5,1,AP,1988,1S


In [4]:
df_datos_hist.shape[0], df_datos_hist["COD_ESTUDIANTE"].nunique()

(502063, 9586)

In [5]:
df_datos_hist.keys()

Index(['ANIO', 'TERMINO', 'IDPERIODO', 'COD_ESTUDIANTE', 'COD_MATERIA_ACAD',
       'PROMEDIO', 'VEZ_TOMADA', 'ESTADO_MAT_TOMADA', 'ANIO_INGRESO',
       'TERMINO_INGRESO'],
      dtype='object')

In [6]:
# mayor a 1
df_datos_hist["COD_ESTUDIANTE"].value_counts()[df_datos_hist["COD_ESTUDIANTE"].value_counts() > 1]

COD_ESTUDIANTE
200820363    124
200712065    117
200822591    114
200617819    114
200123768    114
            ... 
202006821      4
202007142      4
202004297      4
202004602      4
202003935      3
Name: count, Length: 9584, dtype: int64

#### Datos del colegio

In [7]:
df_datos_colegio = pd.read_csv("data/desercion/colegio.csv")

In [8]:
df_datos_colegio["COD_ESTUDIANTE"].nunique(), df_datos_colegio.shape[0]

(9586, 9586)

In [9]:
df_datos_colegio.keys() # en socioeconomico hay beca, pension no todos pagan lo mismo. Categorizar por pension

Index(['COD_ESTUDIANTE', 'PAIS', 'PROVINCIA', 'CANTON', 'PENSION',
       'TIPOCOLEGIO'],
      dtype='object')

#### Nucleo familiar

In [11]:
df_nucleofamiliar_fam_est = pd.read_csv("data/desercion/nucleofamiliar_fam_est.csv")
# parentesco yo mismo

In [12]:
df_nucleofamiliar_fam_est["COD_ESTUDIANTE"].nunique(), df_nucleofamiliar_fam_est.shape[0]

(8320, 34995)

In [13]:
df_nucleofamiliar_fam_est.keys()  

Index(['CODESTUDIANTE', 'COD_ESTUDIANTE', 'IDPERSONA', 'PARENTESCO',
       'TIPOIDENTIFICACION', 'EDAD', 'NIVELINSTRUCCION', 'OCUPACION',
       'INGRESOMENSUAL', 'NIVELAPORTACION', 'TIPOSEGURO', 'TIENEDISCAPACIDAD',
       'ENFERMEDADCASTATROFICA', 'IDENTIFICACION', 'RUTADISCAPACIDAD',
       'RUTAENFERMEDAD', 'RUTAENFERMEDADPREEXISTENTE'],
      dtype='object')

In [ ]:
df_nucleofamiliar_fam_est.head()
# No mucha variabilidad, TIENEDISCAPACIDAD, RUTADISCAPACIDAD	RUTAENFERMEDAD	RUTAENFERMEDADPREEXISTE
# HACER CON INGRESOMENSUAL, OCUPACION

,CODESTUDIANTE,COD_ESTUDIANTE,IDPERSONA,PARENTESCO,TIPOIDENTIFICACION,EDAD,NIVELINSTRUCCION,OCUPACION,INGRESOMENSUAL,NIVELAPORTACION,TIPOSEGURO,TIENEDISCAPACIDAD,ENFERMEDADCASTATROFICA,IDENTIFICACION,RUTADISCAPACIDAD,RUTAENFERMEDAD,RUTAENFERMEDADPREEXISTENTE
0,201305056,201305056,645537,Madre,CEDULA,69,Primaria Completa,Desocupados/Desempleado,0,NINGUNO,Ninguno de los anteriores,NaN,NaN,0908332448,no,no,no
1,201305056,201305056,645537,Hermano(a),CEDULA,47,Secundaria Completa,Desocupados/Desempleado,0,NINGUNO,Ninguno de los anteriores,NaN,NaN,0917765653,no,no,no
2,201305056,201305056,645537,Otro,CEDULA,39,Secundaria Completa,"Cuenta propia (Incluye vendedores ambulantes, ...",394,MEDIO,Ninguno de los anteriores,NaN,NaN,0925615304,no,no,no
3,201305056,201305056,645537,Hermano(a),CEDULA,50,Secundaria Completa,"Cuenta propia (Incluye vendedores ambulantes, ...",400,ALTO,Ninguno de los anteriores,NaN,NaN,0916191158,no,no,no
4,201305056,201305056,645537,Yo mismo (estudiante),CEDULA,29,Superior no Universitaria (tecnología o técnic...,ESTUDIAR,425,ALTO,Ninguno de los anteriores,NaN,NaN,0930479589,no,no,no


In [105]:
df_nucleofamiliar_fam_est.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34995 entries, 0 to 34994
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   CODESTUDIANTE               34995 non-null  int64 
 1   COD_ESTUDIANTE              34995 non-null  int64 
 2   IDPERSONA                   34995 non-null  int64 
 3   PARENTESCO                  34995 non-null  object
 4   TIPOIDENTIFICACION          34875 non-null  object
 5   EDAD                        34995 non-null  int64 
 6   NIVELINSTRUCCION            34988 non-null  object
 7   OCUPACION                   34981 non-null  object
 8   INGRESOMENSUAL              34995 non-null  int64 
 9   NIVELAPORTACION             34959 non-null  object
 10  TIPOSEGURO                  34962 non-null  object
 11  TIENEDISCAPACIDAD           1892 non-null   object
 12  ENFERMEDADCASTATROFICA      1106 non-null   object
 13  IDENTIFICACION              34994 non-null  ob

In [106]:
df_nucleofamiliar_fam_est.describe(include='all')

,CODESTUDIANTE,COD_ESTUDIANTE,IDPERSONA,PARENTESCO,TIPOIDENTIFICACION,EDAD,NIVELINSTRUCCION,OCUPACION,INGRESOMENSUAL,NIVELAPORTACION,TIPOSEGURO,TIENEDISCAPACIDAD,ENFERMEDADCASTATROFICA,IDENTIFICACION,RUTADISCAPACIDAD,RUTAENFERMEDAD,RUTAENFERMEDADPREEXISTENTE
count,3.499500e+04,3.499500e+04,34995.000000,34995,34875,34995.000000,34988,34981,34995.000000,34959,34962,1892,1106,34994,34995,34995,34995
unique,NaN,NaN,NaN,15,2,NaN,11,13,NaN,4,10,6,9,32844,2,2,2
top,NaN,NaN,NaN,Hermano(a),CEDULA,NaN,Superior Universitaria incompleta,ESTUDIAR,NaN,NINGUNO,Ninguno de los anteriores,No tiene discapacidad,No tiene enfermedad,0000000000,no,no,no
freq,NaN,NaN,NaN,10035,34830,NaN,9442,15165,NaN,19506,23762,1340,753,22,34334,34580,34634
mean,2.016730e+08,2.016730e+08,364015.381140,NaN,NaN,34.877983,NaN,NaN,288.425918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2.820653e+05,2.820653e+05,280100.486821,NaN,NaN,18.355948,NaN,NaN,535.439028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.996100e+08,1.996100e+08,4080.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2.015098e+08,2.015098e+08,97171.000000,NaN,NaN,21.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2.017094e+08,2.017094e+08,102842.000000,NaN,NaN,27.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2.019055e+08,2.019055e+08,652672.000000,NaN,NaN,51.000000,NaN,NaN,450.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [109]:
df_nucleofamiliar_fam_est["RUTAENFERMEDADPREEXISTENTE"].value_counts()

RUTAENFERMEDADPREEXISTENTE
no    34634
si      361
Name: count, dtype: int64

In [15]:
df_nucleofamiliar_fam_est["PARENTESCO"].value_counts()

PARENTESCO
Hermano(a)               10035
Yo mismo (estudiante)     8287
Madre                     7393
Padre                     5699
Abuelo/a                  1049
Otro                       662
Tío/a                      484
Hijo(a)                    472
Pareja                     292
PRIMO(A                    251
PADRASTRO                  218
HERMANASTRO                 88
MADRASTRA                   49
HIJASTRO(A)                 12
Padrino/Madrina              4
Name: count, dtype: int64

In [110]:
df_nucleofamiliar_fam_est["OCUPACION"].value_counts()

OCUPACION
ESTUDIAR                                                                                                                            15165
Empleado privado                                                                                                                     5057
QUEHACERES DOMÉSTICOS                                                                                                                3563
Desocupados/Desempleado                                                                                                              3175
Cuenta propia (Incluye vendedores ambulantes, comerciante informal, comercial formal (con RUC activo), sin trabajadores a cargo)     3044
Empleado de Gobierno/Estado                                                                                                          2093
Jubilado                                                                                                                             1137
Jornalero o peón        

#### Merge de datos historico, colegio y nucleo familiar

In [ ]:
# df_datos_hist["COD_ESTUDIANTE"] merge df_datos_colegio["COD_ESTUDIANTE"]
res_merge = pd.merge(df_datos_hist, df_datos_colegio, how="left", on="COD_ESTUDIANTE")
res_merge.shape[0], res_merge["COD_ESTUDIANTE"].nunique()

(502063, 9586)

In [16]:
res_merge.head()

,ANIO,TERMINO,IDPERIODO,COD_ESTUDIANTE,COD_MATERIA_ACAD,PROMEDIO,VEZ_TOMADA,ESTADO_MAT_TOMADA,ANIO_INGRESO,TERMINO_INGRESO,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO
0,2017,1S,564,201614807,ACUG1003,6.00,1,AP,2016,2S,ECUADOR,GUAYAS,GUAYAQUIL,0.72,Nacional
1,2017,1S,564,201413602,ACUG1001,5.74,1,AN,2014,2S,ECUADOR,MANABI,PORTOVIEJO,55.00,Particular
2,2017,1S,564,201248595,ACUG1001,6.01,1,AP,2012,2S,ECUADOR,GUAYAS,GUAYAQUIL,0.72,Fiscal
3,2017,1S,564,201314211,ACUG1001,6.03,1,AP,2013,2S,ECUADOR,GUAYAS,GUAYAQUIL,20.00,Particular
4,2017,1S,564,201408619,ACUG1001,6.06,1,AP,2014,1S,ECUADOR,GUAYAS,GUAYAQUIL,31.00,Particular


In [17]:
res_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502063 entries, 0 to 502062
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ANIO               502063 non-null  int64  
 1   TERMINO            502063 non-null  object 
 2   IDPERIODO          502063 non-null  int64  
 3   COD_ESTUDIANTE     502063 non-null  int64  
 4   COD_MATERIA_ACAD   502063 non-null  object 
 5   PROMEDIO           502063 non-null  float64
 6   VEZ_TOMADA         502063 non-null  int64  
 7   ESTADO_MAT_TOMADA  502063 non-null  object 
 8   ANIO_INGRESO       502063 non-null  int64  
 9   TERMINO_INGRESO    502063 non-null  object 
 10  PAIS               502063 non-null  object 
 11  PROVINCIA          502063 non-null  object 
 12  CANTON             502063 non-null  object 
 13  PENSION            502063 non-null  float64
 14  TIPOCOLEGIO        502063 non-null  object 
dtypes: float64(2), int64(5), object(8)
memory usage: 57

In [18]:
res_merge.describe()

,ANIO,IDPERIODO,COD_ESTUDIANTE,PROMEDIO,VEZ_TOMADA,ANIO_INGRESO,PENSION
count,502063.000000,502063.000000,5.020630e+05,502063.000000,502063.000000,502063.000000,502063.000000
mean,2018.684910,555.869988,2.015893e+08,7.124491,1.109225,2015.820718,99.091229
std,3.204816,76.535119,3.334555e+05,1.880328,0.353910,3.317908,122.888148
min,1999.000000,1.000000,1.988024e+08,0.000000,0.000000,1988.000000,0.000000
25%,2017.000000,564.000000,2.014179e+08,6.340000,1.000000,2014.000000,0.720000
50%,2019.000000,572.000000,2.016109e+08,7.360000,1.000000,2016.000000,55.000000
75%,2021.000000,581.000000,2.018105e+08,8.400000,1.000000,2018.000000,159.000000
max,2025.000000,603.000000,2.020500e+08,10.000000,4.000000,2020.000000,748.610000


In [64]:
res_merge["COD_ESTUDIANTE"].nunique()

9586

#### Socioeconomico

In [97]:
df_socio = pd.read_csv("data/desercion/socioeconomico_18913.csv")

C:\Users\saraujo\AppData\Local\Temp\ipykernel_33444\2623598426.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_socio = pd.read_csv("data/desercion/socioeconomico_18913.csv")


In [98]:
df_socio.shape

(18913, 131)

In [99]:
df_socio.head()

,ESTACOMPLETA,IDPERSONA,FECHACREACION,FECHAENVIOUBEP,FECHAENVIO,CODESTUDIANTE,APELLIDOS,NOMBRES,EMAIL,P1_NACIONALIDAD,NUMEROIDENTIFICACION,ANIO_TERMINO_INGRESO,CATEGORIA,ISE,TIENEDISCAPACIDAD,TIPODISCAPACIDAD,PORCENTAJEDISCAPACIDAD,SEXO,AUTOIDENTIFICACIONGENERO,AUTOIDENTIFICACIONETNICA,ESTADOCIVIL,FECHANACIMIENTO,PAISNACIMIENTO,PROVINCIANACIMIENTO,CIUDADNACIMIENTO,TELEFONOCELULAR,TELEFONOFIJO,CORREOALTERNO,COLEGIO,PAISCOLEGIO,PROVINCIACOLEGIO,CANTONCOLEGIO,TIPOCOLEGIO,ANIOGRADUACION,CATEGORIACOLEGIO,JORNADA,BECACOLEGIO,OTROSIDIOMAS,IDIOMAS,COBOCIMIENTOINGLESPOR,COMIDASALDIA,FRASETRESCOMIDAS,FRASEHABITOALIMENTICIO,FRASEHABITOALIMENTICIOV2,CONSUMODIARIOESPOL,TIEMPOPROMEDIOLLEGARESPOL,VECESBUSENTRADA,VECESCARROENTRADA,BICICLETAENTRADA,TIEMPOPROMEDIOBICICLETAENTRADAESPOL,VECESTAXIENTRADA,VECESCARROMOTOAMIGOENTRADA,CAMINAENTRADA,TIEMPOPROMEDIOCAMINATAENTRADAESPOL,VECESTRICIMOTOENTRADA,VECESBUSSALIDA,VECESCARROSALIDA,BICICLETASALIDA,TIEMPOPROMEDIOBICICLETASALIDAESPOL,VECESTAXISALIDA,VECESCARROMOTOAMIGOSALIDA,CAMINASALIDA,TIEMPOPROMEDIOCAMINATASALIDAESPOL,VECESTRICIMOTOSALIDA,NIVELINGLES,POSEETARJETACREDITO,POSEETARJETADEBITO,CUENTASBANCO,HERMANOSESTUDIANDOESPOL,ALIMENTACION,TRANSPORTE,SERVICIOS,ARRIENDO,ALICUOTAS,VESTIMENTA,SALUD,EDUCACION,TARJETACREDITO,ENTRETENIMIENTO,OTROS,NIVELINSTRUCCIONPADRE,NIVELINSTRUCCIONMADRE,ESTADOCIVILPADRES,DISCAPACIDAD,FAMILIARDISCAPACIDAD,ENFERMEDAD,FAMILIARENFERMEDAD,RECIBEBONO,DIFICULTADAPRENDIZAJE,PAISVIVE,PROVINCIAVIVE,CIUDADVIVE,PARROQUIAVIVE,DIRECCION,COORDENADAS,TIPOPARROQUIA,VIVEGRUPOFAMILIAR,PAISVIVESEP,PROVINCIAVIVIENDASEP,CANTONVIVESEP,PARROQUIAVIVESEP,DIRECCIONVIVSEP,TIPOVIVIENDASEP,ESTADOVIVIENDASEP,CANTIDADCUARTOS,CANTIDADBANIO,SALA,COMEDOR,ESTUDIO,COCINA,LAVANDERIA,GARAJE,METERIALTECHOVIVIENDASEP,METERIALPISOVIVIENDASEP,METERIALPAREDVIVIENDASEP,VIAACCESOVIVIENDASEP,ABASTECIMIENTOAGUA,SERVHIGIENE,ELIMINACIONBASURA,SERVICIOELECTRICIDAD,POSEEVEHICULO,CANTIDADVEHICULO,RECIBEAYUDA,PARIENTEAYUDA,VALORAYUDA,TIPOBACHILLER,COMIDAS,DISPOSITIVOS,MANEJOCELULAR,ACCESOINTERNET,NUMEROSFAMILIARES
0,1,7897,2020-05-12,NaN,2020-10-08-10.32.44.471004,199900572,TRUJILLO MENESES,ROBERTO SENEN,rtrujill@espol.edu.ec,ECUATORIANA,911235539,1999 1S,6.0,2.0000,N,NaN,0.0,Masculino,masculino,Mestizo,divorciado,1979-09-08,ECUADOR,GUAYAS,GUAYAQUIL,84009858,2857385; 381377,truckson@hotmail.com,UNIDAD EDUCATIVA EXPERIMENTAL URDESA SCHOOL G...,ECUADOR,GUAYAS,GUAYAQUIL,Instituto,1999.0,0.0,MATUTINA (07:00-14:00 APROX.),Ninguna,SI,INGLÉS,Aprendizaje del colegio,3,NaN,Salgo de ESPOL y voy a comer a casa,NaN,0,31 a 60 minutos,0,1,0,NaN,0,0,0,NaN,-1,0,1,0,NaN,0,0,0,NaN,-1,NaN,NO,NO,NaN,NO,400,150,75,600,100,200,1,900,0,0,0,NaN,NaN,Divorciados,NO,NaN,NO,NaN,NO,NaN,ECUADOR,GUAYAS,GUAYAQUIL,XIMENA,COSTANERA DEL SALADO 615,NaN,URBANA,SI,NaN,NaN,NaN,NaN,NaN,NaN,Arriendo O Alquiler,3,4,SI,SI,NO,SI,SI,SI,Hormigón/losa/cemento,Cerámica / Baldosa / Vinil,Hormigón / ladrillo / bloque / cemento,Carretera / calle pavimentada o adoquinada / c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1
1,1,44424,2021-05-29,NaN,2021-05-17-09.08.30.243167,200212538,VERA AGUILAR,CINTHYA VANESSA,cvvera@espol.edu.ec,ECUATORIANA,921930699,2002 1S,4.0,0.7913,N,no definida,0.0,Femenino,femenino,Mestizo,casado,1984-08-17,ECUADOR,GUAYAS,GUAYAQUIL,99528609,2157540,cinthyavera84@gmail.com,UNIDAD EDUCATIVA SANTA MARIA MAZZARELLO GUAYAQUIL,ECUADOR,GUAYAS,GUAYAQUIL,Fiscomisional,2002.0,1.0,MATUTINA (07:00-14:00 APROX.),Deportiva o cultural,SI,FRANCÉS; INGLÉS,Cursos en academias locales,3,NaN,Salgo de ESPOL y voy a comer a casa,NaN,0,16 a 30 minutos,0,1,0,NaN,0,0,0,NaN,-1,0,1,0,NaN,0,0,0,NaN,-1,NaN,SI,NaN,NaN,NO,300,200,100,400,30,100,100,600,0,100,0,NaN,NaN,Casados,NO,NaN,NO,NaN,NO,NaN,ECUADOR,GUAYAS,GUAYAQUIL,TARQUI,CIUDAD DEL RIO II MZ 946 VILLA 20,"-2.061471649983754,-79.91365241483554",URBANA,SI,NaN,NaN,NaN,NaN,NaN,NaN,Arriendo O Alquiler,2,2,SI,SI,NO,SI,SI,NO,Hormigón/losa/cemento,Cerámica / Baldosa / Vinil,Hormigón / ladrillo / bloque / cemento,C

In [100]:
# type int
# df_socio["CODESTUDIANTE"] = df_socio["CODESTUDIANTE"].str.strip()
df_socio["CODESTUDIANTE"] = df_socio["CODESTUDIANTE"].astype(int)

In [101]:
res_merge["COD_ESTUDIANTE"].dtype, df_socio["CODESTUDIANTE"].dtype

(dtype('int64'), dtype('int64'))

In [102]:
res_merge["COD_ESTUDIANTE"].nunique(), df_socio["CODESTUDIANTE"].nunique()

(9586, 18804)

In [103]:
# los que no estan en socioeconomico
res_merge[~res_merge["COD_ESTUDIANTE"].isin(df_socio["CODESTUDIANTE"])]["COD_ESTUDIANTE"].nunique()

1266

In [104]:
res_merge[~res_merge["COD_ESTUDIANTE"].isin(df_socio["CODESTUDIANTE"])]["COD_ESTUDIANTE"].unique()

array([201405193, 201514161, 201420603, ..., 201904992, 201910734,
       201916558])